In [1]:
!pip install -q langgraph langchain-google-genai langchain-community duckduckgo-search ddgs beautifulsoup4 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 80.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
import os, getpass

if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [4]:
import requests
from bs4 import BeautifulSoup
from langchain_core.tools import tool

@tool
def jd_fetcher(url: str) -> str:
    """
    Fetch a job description from a URL and return clean plain text.

    Use ONLY when the user provides a job posting URL.

    Returns first 4000 characters of cleaned page text.
    """

    try:

        r = requests.get(
            url,
            headers={'User-Agent': 'Mozilla/5.0'},
            timeout=10
        )

        r.raise_for_status()

        soup = BeautifulSoup(r.text, 'html.parser')

        for tag in soup(['script', 'style']):
            tag.decompose()

        return soup.get_text(
            separator='\n',
            strip=True
        )[:4000]

    except Exception as e:

        return f'ERROR: failed to fetch URL — {e}'

In [5]:
print(
    jd_fetcher.invoke({
        'url': 'https://example.com'
    })[:500]
)

Example Domain
Example Domain
This domain is for use in documentation examples without needing permission. Avoid use in operations.
Learn more


In [6]:
@tool
def skills_gap(student_skills: str,
               must_have_skills: str) -> str:

    """
    Compare student skills against required job skills.

    Returns missing skills or 'none'.
    """

    a = set(
        s.strip().lower()
        for s in student_skills.split(',')
        if s.strip()
    )

    b = set(
        s.strip().lower()
        for s in must_have_skills.split(',')
        if s.strip()
    )

    missing = sorted(b - a)

    return ', '.join(missing) if missing else 'none'

In [7]:
print(
    skills_gap.invoke({
        'student_skills': 'Python, Java, SQL',
        'must_have_skills': 'Python, Java, SQL, AWS, Spring Boot'
    })
)

aws, spring boot


In [17]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model='gemini-2.0-flash'
)

@tool
def answer_scorer(question: str,
                  answer: str) -> str:

    """
    Score interview answers from 1-10
    with one-line rationale.
    """

    prompt = f'''
Score this placement interview answer 1-10.

Question:
{question}

Answer:
{answer}

Return format:
Score: X/10
Rationale: ...
'''

    return llm.invoke(prompt).content

In [9]:
print(
    answer_scorer.invoke({
        'question': 'Why TCS Digital?',
        'answer': 'Because TCS is big and they pay well.'
    })
)

Score: **2/10**

Rationale:

This answer is extremely weak and unlikely to impress an interviewer.

1.  **Fails to Address "Digital":** The question specifically asks "Why TCS **Digital**?" Your answer completely ignores the "Digital" aspect, which is crucial. TCS Digital focuses on cutting-edge technologies, innovation, client solutions, and transforming businesses. Your answer could apply to almost any large company.
2.  **Generic and Lacks Research:** "Big" is a very generic observation. Many companies are big. It doesn't show you've done any specific research into TCS Digital, its projects, its culture, or its impact.
3.  **Focus on Pay is a Red Flag:** While everyone wants to be paid well, stating it as a primary reason (especially in such a blunt manner) signals to the interviewer that your motivation is purely transactional. Interviewers look for passion, alignment with company values, interest in the work, and a desire to contribute. Someone primarily motivated by pay might be 

In [10]:
from langgraph.prebuilt import create_react_agent

tools = [
    jd_fetcher,
    skills_gap,
    answer_scorer
]

agent = create_react_agent(
    llm,
    tools=tools
)

print(f'Agent created with {len(tools)} tools.')

Agent created with 3 tools.


/tmp/ipykernel_1582/658485430.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [11]:
profiles = [

    {
        "name": "Ravi Kumar",
        "branch": "CSE",
        "cgpa": 8.1,
        "skills": ["Python", "Java", "SQL"],
        "target_company": "TCS"
    },

    {
        "name": "Sneha Reddy",
        "branch": "ECE",
        "cgpa": 8.8,
        "skills": ["Python", "Power BI", "Excel"],
        "target_company": "Cognizant"
    },

    {
        "name": "Arun Pillai",
        "branch": "IT",
        "cgpa": 9.0,
        "skills": ["Python", "AWS", "Docker", "SQL"],
        "target_company": "Amazon"
    }
]

In [14]:
for i, p in enumerate(profiles):

    print(f'\n{"="*60}')
    print(f'Student {i+1}: {p["name"]}')
    print(f'{"="*60}')

    msg = (
    f"I am {p['name']}, "
    f"B.Tech {p['branch']} "
    f"CGPA {p['cgpa']}, "
    f"skills: {', '.join(p['skills'])}. "
    f"Target: {p['target_company']}. "

    f"Use the skills_gap tool to identify missing skills. "

    f"Then use the answer_scorer tool to evaluate this answer: "

    f"'I want to join because the company is famous.' "

    f"Finally generate 3 mock interview questions."
)

    result = agent.invoke(
        {
            'messages': [('user', msg)]
        },
        config={'recursion_limit': 10}
    )

    for j, m in enumerate(result['messages']):

        print(f'\n[{j}] {type(m).__name__}')

        if hasattr(m, 'content') and m.content:
            print(str(m.content)[:300])

        if hasattr(m, 'tool_calls') and m.tool_calls:

            for tc in m.tool_calls:
                print(f'→ tool_call: {tc.get("name")}')


Student 1: Ravi Kumar

[0] HumanMessage
I am Ravi Kumar, B.Tech CSE CGPA 8.1, skills: Python, Java, SQL. Target: TCS. Use the skills_gap tool to identify missing skills. Then use the answer_scorer tool to evaluate this answer: 'I want to join because the company is famous.' Finally generate 3 mock interview questions.

[1] AIMessage
[{'type': 'text', 'text': 'Please provide the must-have skills for TCS for me to identify the missing skills. Also, tell me the question for which the answer "I want to join because the company is famous" was given, so I can score it.', 'extras': {'signature': 'CvoGAQw51seIGVtHftAD8t9zRlv2+y+nwl/JHz

Student 2: Sneha Reddy

[0] HumanMessage
I am Sneha Reddy, B.Tech ECE CGPA 8.8, skills: Python, Power BI, Excel. Target: Cognizant. Use the skills_gap tool to identify missing skills. Then use the answer_scorer tool to evaluate this answer: 'I want to join because the company is famous.' Finally generate 3 mock interview questions.

[1] AIMessage
[{'type': 'te

In [19]:
result = agent.invoke({
    'messages': [
        (
            'user',
            'Fetch this JD and tell me skills: '
            'https://this-does-not-exist-99999.example.com/jd'
        )
    ]
})

for j, m in enumerate(result['messages']):

    print(f'\n[{j}] {type(m).__name__}')

    if hasattr(m, 'content') and m.content:
        print(str(m.content)[:300])


[0] HumanMessage
Fetch this JD and tell me skills: https://this-does-not-exist-99999.example.com/jd

[1] AIMessage
[{'type': 'text', 'text': 'I am sorry, but I am unable to fetch the job description from the provided URL. It appears to be an invalid link. Please double-check the URL and try again.', 'extras': {'signature': 'CvEHAQw51seSNQ9qru54KrowDaxKYWPdzuVNlh27p2fd+DzHDufqHEnDHWRQwusxKLcKINfNgzgJdcxJQheqmca3y


## Day 9 Capstone Sprint 4 — Career Agent

### Tools

1. jd_fetcher
2. skills_gap
3. answer_scorer

### Key Learning

The agent dynamically selects tools using the ReAct loop:

Thought → Action → Observation → Answer

### Failure Recovery

Bad URLs return explicit ERROR strings to prevent hallucination.